# 04 · A helper model for the selection steps

Two later steps search over predictors: notebook 05 weighs near-duplicate predictors, and
notebook 06 grows the parsimonious set one predictor at a time. Both need a model that is
fast enough to refit thousands of times, so the study uses **LightGBM** as a helper, tuned
once per rung on all 95 predictors of the pool. It is a tool for the searches, not one of the
library's models.

Tuning, here and everywhere in the study, is the same procedure:

- **Optuna's TPE sampler** with a fixed seed (2026), a fixed number of trials (60 for LightGBM);
- each trial is scored by **survey-weighted average precision, out of fold**, over the five
  development folds of notebook 03;
- inside each fold the survey weights are rescaled to average one, so their size does not
  interact with the model's regularisation;
- the held-out women are not touched.

**Writes:** `outputs/enhanced/<rung>/best_hyperparameters.json`

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

RETUNE = True   # False: skip the search and use the study's settings (about 45 minutes saved)
study = repro.reference("selector_lightgbm.json")

In [ ]:
for rung in lib.RUNGS:
    d = lib.data("pool95", rung)
    if RETUNE:
        params, best = tuning.tune("LightGBM", d.X_dev, d.y_dev, d.w_dev, d.splits, d.numeric, d.categorical,
                                   n_trials=tuning.TRIALS["LightGBM"])
        same = params == study[rung]
        repro.check(f"{rung}: the search finds the study's settings", same, "" if same else f"found {params}")
        use = study[rung] if (FOLLOW_STUDY and not same) else params
    else:
        use = study[rung]
    out = P.outputs / "enhanced" / rung
    out.mkdir(parents=True, exist_ok=True)
    json.dump({"LightGBM": use}, open(out / "best_hyperparameters.json", "w"), indent=2)
    print(rung, use)
repro.report()